# Tier 1 Evaluation — MSR-VTT 1k-A Zero-Shot Text-to-Video Retrieval

## Purpose

VideoRAG 프로토타입의 검색 성능을 **표준 벤치마크(MSR-VTT 1k-A)**로 정량 평가.

## Protocol

| Item | Value |
|---|---|
| **Benchmark** | MSR-VTT 1k-A (JSFusion split, Yu et al. 2018) |
| **Corpus** | 1,000 test videos (video7010 ~ video9999 중 지정된 1000개) |
| **Query** | 1 designated caption per video × 1,000 |
| **Ground Truth** | query의 source video (1:1 매핑) |
| **Index** | FAISS IndexFlatIP — exact brute-force cosine (eval 전용) |
| **Metrics** | R@1, R@5, R@10, Median Rank (MdR), Mean Rank (MnR) |

## Paper Baseline (InternVideo2, Table 24a Supplementary)

| Model | #Frames | R@1 | R@5 | R@10 |
|---|---|---|---|---|
| InternVideo2s2-1B | 4 | **51.9** | **74.6** | **81.7** |
| InternVideo2s2-1B | 8 | 51.9 | 75.3 | 82.5 |

Our embedder: `OpenGVLab/InternVideo2-Stage2_1B-224p-f4` (1B, 4 frames, 512-dim joint space)

## Evaluation Plan

- **Tier 1**: Dense-only on 1k-A → R@1/5/10 vs paper baseline (±2-3% = valid reproduction)
- **Tier 1.5**: Full pipeline latency ablation (BM25 / Dense / Hybrid / +ColBERT) on production corpus

> 이 노트북은 **독립 실행 가능**합니다 — Drive 마운트, 의존성 설치가 Step 0에 포함.
> 단, `01_indexing.ipynb`로 production 인덱스가 빌드되어 있어야 합니다 (Tier 1.5 용).

## VideoRAG 벤치마크 평가 노트북 설명

이 문서는 **"우리가 만든 VideoRAG 시스템이 얼마나 검색을 잘 하는지 객관적으로 측정하자"** 는 실험 계획서야.

---

### 🎯 Purpose (목적)

VideoRAG 프로토타입의 검색 성능을 **표준 벤치마크**로 측정하겠다는 것.

> 벤치마크란? → 누구나 공인된 동일한 시험지로 점수를 내서 비교할 수 있게 해주는 데이터셋이야. 수능처럼.

사용하는 벤치마크는 **MSR-VTT 1k-A** — Microsoft Research에서 만든 영상-텍스트 검색용 표준 시험지.

---

### 📋 Protocol (실험 세팅)

| 항목 | 의미 |
|---|---|
| **Benchmark** | MSR-VTT 1k-A, JSFusion split (Yu et al. 2018) — 어떤 버전의 시험지를 쓸지 |
| **Corpus** | 1,000개의 테스트 영상 (video7010~video9999 중 1000개) — 검색 대상 풀 |
| **Query** | 각 영상마다 지정된 캡션 1개 × 1,000개 — "이 설명으로 영상을 찾아봐" 하는 질문 |
| **Ground Truth** | 각 query의 정답 영상이 1:1로 정해져 있음 |
| **Index** | FAISS IndexFlatIP (완전 브루트포스 코사인) — 평가 전용으로 가장 정확한 방식 사용 |
| **Metrics** | R@1, R@5, R@10, MdR, MnR |

#### Metrics가 뭔지 풀어서 설명하면

- **R@1** = 1,000개 영상 중 **딱 1개만 뽑았을 때** 정답이 포함될 확률
- **R@5** = **5개 뽑았을 때** 정답이 포함될 확률
- **R@10** = **10개 뽑았을 때** 정답이 포함될 확률
- **MdR (Median Rank)** = 정답이 몇 번째에 나오는지의 중앙값 (낮을수록 좋음)
- **MnR (Mean Rank)** = 정답 순위의 평균값 (낮을수록 좋음)

---

### 📊 Paper Baseline

우리가 사용하는 임베더 모델(**InternVideo2s2-1B**)이 논문에서 공식 발표한 성능이야.

| 모델 | 프레임 수 | R@1 | R@5 | R@10 |
|---|---|---|---|---|
| InternVideo2s2-1B | 4 | **51.9** | **74.6** | **81.7** |
| InternVideo2s2-1B | 8 | 51.9 | 75.3 | 82.5 |

우리가 쓰는 임베더: `OpenGVLab/InternVideo2-Stage2_1B-224p-f4`
- 1B = 10억 파라미터 모델
- 4 frames = 영상에서 4장의 프레임 추출해서 임베딩
- 512-dim = 512차원 벡터로 영상과 텍스트를 같은 공간에 표현

---

### 🗺️ Evaluation Plan

#### Tier 1 (기본 검증)
Dense 임베딩만 써서 1k-A로 돌렸을 때 **논문 수치 ±2~3% 이내**로 재현되면 OK.
→ "우리 구현이 올바른가?" 확인하는 단계

#### Tier 1.5 (파이프라인 비교)
실제 프로덕션 코퍼스에서 4가지 방식의 **속도/성능 비교**:
- BM25 (키워드 검색)
- Dense (벡터 검색)
- Hybrid (BM25 + Dense)
- +ColBERT (리랭킹 추가)

---

### 💡 한 줄 요약

> **"우리 VideoRAG가 논문만큼 잘 작동하는지 공인 시험으로 검증하고, 더 나아가 BM25/Dense/Hybrid/ColBERT 각 방식의 속도와 성능을 실제 환경에서 비교해보자"** 는 실험 계획서야.

In [ ]:
# ── Step 0: 환경 부트스트랩 (런타임 초기화 시 1회) ──

import os, shutil
os.chdir('/content')
from google.colab import userdata

# (1) Google Drive 마운트
if not os.path.exists('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
    print('✓ Drive 마운트 완료')
else:
    print('✓ Drive 이미 마운트됨')

# (2) 프로젝트 코드 복사 (Drive → 로컬)
DRIVE_PROJECT = '/content/drive/MyDrive/videorag_prototype'
LOCAL_PROJECT = '/content/videorag_prototype'

if not os.path.exists(LOCAL_PROJECT):
    if os.path.exists(DRIVE_PROJECT):
        shutil.copytree(DRIVE_PROJECT, LOCAL_PROJECT)
        print('✓ Drive → 로컬 복사 완료')
    else:
        print(f'⚠ {DRIVE_PROJECT} 없음 → git clone 필요')
else:
    print(f'✓ 프로젝트 이미 존재: {LOCAL_PROJECT}')

# (2-1) GitHub에서 최신 src 동기화
# [Public repo] token 불필요 — public clone
os.system(f"rm -rf /tmp/VideoRAG-Prototype")
os.system("git clone https://github.com/LimPark996/VideoRAG-Public.git /tmp/VideoRAG-Prototype")
os.chdir('/content')
os.system(f"rm -rf {LOCAL_PROJECT}/src")
os.system(f"cp -r /tmp/VideoRAG-Prototype/src {LOCAL_PROJECT}/src")
os.system(f"rm -rf {LOCAL_PROJECT}/notebooks")
os.system(f"cp -r /tmp/VideoRAG-Prototype/notebooks {LOCAL_PROJECT}/notebooks")
print('✓ src + notebooks 클론 완료')

# (3) 핵심 의존성 설치
!pip install -q open_clip_torch 2>/dev/null
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118 2>/dev/null
!pip install -q transformers timm einops rank_bm25 faiss-cpu \
    moviepy opencv-python cryptography scikit-learn scipy \
    tqdm matplotlib pandas numpy easydict langdetect requests 2>/dev/null
!pip install -q ragatouille colbert-ai 2>/dev/null
!git clone -q https://github.com/soCzech/TransNetV2 /content/TransNetV2 2>/dev/null
!pip install -q -e /content/TransNetV2/ 2>/dev/null
print('✓ 의존성 확인 완료')

# (4) sys.path 등록
import sys
sys.path.insert(0, LOCAL_PROJECT)
sys.path.insert(0, '/content/TransNetV2/inference')
print(f'✓ sys.path에 {LOCAL_PROJECT} 추가')

# (5) HF_TOKEN 설정
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
print('✓ HF_TOKEN 설정 완료')

# (6) Papago 설정
os.environ["PAPAGO_CLIENT_ID"] = userdata.get('PAPAGO_CLIENT_ID')
os.environ["PAPAGO_CLIENT_SECRET"] = userdata.get('PAPAGO_CLIENT_SECRET')
if os.environ["PAPAGO_CLIENT_ID"]:
    print('✓ Papago 설정 완료')
else:
    print('⚠ Papago 자격증명 없음 — 한국어 쿼리 자동 번역 불가')

# (7) 인덱스 존재 확인 (Tier 1.5 latency ablation 용)
INDEX_DIR = os.path.join(LOCAL_PROJECT, 'index')
if os.path.exists(INDEX_DIR) and os.listdir(INDEX_DIR):
    print(f'✓ Production 인덱스 발견: {os.listdir(INDEX_DIR)}')
else:
    DRIVE_INDEX = '/content/drive/MyDrive/videorag_prototype/index'
    if os.path.exists(DRIVE_INDEX) and os.listdir(DRIVE_INDEX):
        os.makedirs(INDEX_DIR, exist_ok=True)
        for f in os.listdir(DRIVE_INDEX):
            shutil.copy2(os.path.join(DRIVE_INDEX, f), os.path.join(INDEX_DIR, f))
        print('✓ Drive에서 인덱스 복원 완료')
    else:
        print('⚠ Production 인덱스 없음 — Tier 1.5는 스킵됩니다')

# (8) 1k-A 테스트 영상 경로 확인
TEST_VIDEO_CANDIDATES = [
    os.path.join(LOCAL_PROJECT, 'data', 'msrvtt', 'test_1ka_videos'),
    '/content/drive/MyDrive/msrvtt/TestVideo',
    '/content/drive/MyDrive/videorag_prototype/data/msrvtt/test_1ka_videos',
    os.path.join(LOCAL_PROJECT, 'data', 'msrvtt', 'videos'),
]
TEST_VIDEO_DIR = None
for path in TEST_VIDEO_CANDIDATES:
    if os.path.exists(path):
        mp4s = [f for f in os.listdir(path) if f.endswith('.mp4')]
        if len(mp4s) >= 100:
            TEST_VIDEO_DIR = path
            print(f'✓ 테스트 영상 발견: {path} ({len(mp4s)}개)')
            break
if TEST_VIDEO_DIR is None:
    print('⚠ 1k-A 테스트 영상 미발견 — Step 3에서 임베딩 캐시 필요')
    print('  영상 위치 후보:')
    for p in TEST_VIDEO_CANDIDATES:
        print(f'    {p}')

In [ ]:
# ── Step 0.5: InternVideo2 설치 (런타임 초기화 시 1회) ──
import shutil, os, sys, importlib

if os.path.exists('/content/InternVideo'):
    shutil.rmtree('/content/InternVideo')
    print('✓ 기존 /content/InternVideo 삭제 완료')

!git clone --no-checkout --depth=1 https://github.com/OpenGVLab/InternVideo.git /content/InternVideo
%cd /content/InternVideo
!git sparse-checkout init --cone
!git sparse-checkout set InternVideo2/multi_modality
!git checkout main

INTERNVIDEO_PATH = '/content/InternVideo/InternVideo2/multi_modality'
if INTERNVIDEO_PATH not in sys.path:
    sys.path.insert(0, INTERNVIDEO_PATH)
importlib.invalidate_caches()
print(f'✓ InternVideo2 설치 완료: {INTERNVIDEO_PATH}')

In [ ]:
# ── Step 1: 파이프라인 초기화 + Parity Check ──
import sys, os, json, time, pickle
import numpy as np

PROJECT_ROOT = '/content/videorag_prototype'
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

INDEX_DIR  = os.path.join(PROJECT_ROOT, 'index')
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

from src.pipeline import VideoRAGPipeline

config = {
    "embed_dim": 512,
    "w_visual": 0.6,
    "w_text": 0.4,
    "rrf_k": 60,
    "papago_client_id": os.environ.get("PAPAGO_CLIENT_ID", ""),
    "papago_client_secret": os.environ.get("PAPAGO_CLIENT_SECRET", ""),
}

pipeline = VideoRAGPipeline(
    index_dir=INDEX_DIR,
    output_dir=OUTPUT_DIR,
    config=config
)

# Production 인덱스 로드 (Tier 1.5 용, 없으면 skip)
HAS_PRODUCTION_INDEX = os.path.exists(INDEX_DIR) and len(os.listdir(INDEX_DIR)) > 0
if HAS_PRODUCTION_INDEX:
    pipeline.load_index()
    print(f'✓ Production 인덱스 로드 완료')
else:
    print('⚠ Production 인덱스 없음 — Tier 1.5 latency ablation 스킵')

# ── Parity Check: InternVideo2 셋팅이 논문과 일치하는지 자가진단 ──
from src.evaluation.faiss_flat_eval import parity_check
parity_report = parity_check(
    pipeline.embedder,
    pipeline.dense_retriever if HAS_PRODUCTION_INDEX else None
)
print('\n✓ 파이프라인 초기화 + Parity Check 완료')

In [ ]:
# ── Step 2: 1k-A Annotation 다운로드 & 로드 ──
import urllib.request

ANN_DIR  = os.path.join(PROJECT_ROOT, 'data', 'msrvtt', 'annotations')
ANN_PATH = os.path.join(ANN_DIR, 'msrvtt_test_1k.json')
os.makedirs(ANN_DIR, exist_ok=True)

# HuggingFace에서 1k-A annotation 다운로드 (1회)
if not os.path.exists(ANN_PATH):
    url = "https://huggingface.co/datasets/friedrichor/MSR-VTT/raw/main/msrvtt_test_1k.json"
    print(f'Downloading 1k-A annotation from HuggingFace...')
    urllib.request.urlretrieve(url, ANN_PATH)
    print(f'✓ 다운로드 완료: {ANN_PATH}')
else:
    print(f'✓ Annotation 이미 존재: {ANN_PATH}')

with open(ANN_PATH, 'r') as f:
    ann_1ka = json.load(f)

# (caption, video_id) 쌍 추출
eval_pairs = [(entry['caption'], entry['video_id']) for entry in ann_1ka]
video_ids_1ka = sorted(set(vid for _, vid in eval_pairs))

print(f'✓ 1k-A eval pairs: {len(eval_pairs)}개')
print(f'  고유 video_id: {len(video_ids_1ka)}개')
print(f'  video_id 범위: {video_ids_1ka[0]} ~ {video_ids_1ka[-1]}')
print(f'\n  예시:')
for cap, vid in eval_pairs[:3]:
    print(f'    "{cap[:50]}..." → {vid}')

In [ ]:
# ── MSR-VTT 주요 영상 zip 다운로드 → 1k-A test만 추출 → Drive 백업 ──
import os, shutil, zipfile, urllib.request

TEST_VIDEO_DIR = '/content/videorag_prototype/data/msrvtt/videos'
DRIVE_BACKUP   = '/content/drive/MyDrive/videorag_prototype/data/msrvtt/videos'
ZIP_PATH       = '/content/MSRVTT_Videos.zip'
os.makedirs(TEST_VIDEO_DIR, exist_ok=True)
os.makedirs(DRIVE_BACKUP, exist_ok=True)

# (1) zip 다운로드 (2.19 GB, T4 기준 1~2분)
url = "https://huggingface.co/datasets/friedrichor/MSR-VTT/resolve/main/MSRVTT_Videos.zip"
if not os.path.exists(ZIP_PATH):
    print(f'다운로드 중... (2.19 GB)')
    urllib.request.urlretrieve(url, ZIP_PATH)
    print(f'✓ {ZIP_PATH}')

# (2) 1k-A test video ID만 추출 (video_ids_1ka는 Cell 5에서 정의)
need = set(video_ids_1ka)
extracted = 0
with zipfile.ZipFile(ZIP_PATH) as zf:
    for member in zf.namelist():
        name = os.path.basename(member)
        if not name.endswith('.mp4') or name[:-4] not in need:
            continue
        dst = os.path.join(TEST_VIDEO_DIR, name)
        if os.path.exists(dst) and os.path.getsize(dst) > 0:
            continue
        with zf.open(member) as src, open(dst, 'wb') as out:
            shutil.copyfileobj(src, out)
        extracted += 1
print(f'✓ 1k-A test 영상 추출: {extracted}개')

# (3) Drive 백업 (1k-A만)
backed = 0
for name in os.listdir(TEST_VIDEO_DIR):
    if not name.endswith('.mp4') or name[:-4] not in need:
        continue
    src = os.path.join(TEST_VIDEO_DIR, name)
    dst = os.path.join(DRIVE_BACKUP, name)
    if not os.path.exists(dst):
        shutil.copy2(src, dst)
        backed += 1
print(f'✓ Drive 백업: {backed}개 → {DRIVE_BACKUP}')

# (4) 1k-A 검증
have = {f[:-4] for f in os.listdir(TEST_VIDEO_DIR) if f.endswith('.mp4')}
missing = sorted(need - have)
print(f'✓ 1k-A: need={len(need)}, matched={len(need & have)}, missing={len(missing)}')
if not missing:
    print('  → Cell 7 진행 가능')

# (선택) zip 삭제로 디스크 회수
# os.remove(ZIP_PATH)

In [ ]:
# ── Step 3: 1k-A 영상 임베딩 + FlatEvalStore 빌드 ──
#
# 1000개 테스트 영상을 InternVideo2로 인코딩하여 eval 전용 exact-search 인덱스 생성.
# 임베딩은 pkl로 캐시 — 다음 실행 시 재인코딩 불필요.
# T4 기준 약 30~60분 소요 (최초 1회만).

import cv2
from tqdm import tqdm
from src.evaluation.faiss_flat_eval import FlatEvalStore, build_flat_index_from_embeddings

EMBED_CACHE = os.path.join(ANN_DIR, 'test_1ka_embeddings.pkl')
EVAL_INDEX_PATH = os.path.join(ANN_DIR, 'test_1ka_flat.index')

def extract_uniform_frames(video_path, num_frames=4):
    """영상에서 균등 간격으로 num_frames장 프레임 추출."""
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return None

    indices = np.linspace(0, total - 1, num_frames, dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames if len(frames) == num_frames else None


# ── 임베딩 로드 or 생성 ──
if os.path.exists(EMBED_CACHE):
    with open(EMBED_CACHE, 'rb') as f:
        video_vecs = pickle.load(f)
    print(f'✓ 캐시에서 임베딩 로드: {len(video_vecs)}개')

elif TEST_VIDEO_DIR is not None:
    print(f'임베딩 생성 시작 (영상 경로: {TEST_VIDEO_DIR})')
    print(f'  대상: {len(video_ids_1ka)}개 영상, num_frames={pipeline.embedder.num_frames}')
    pipeline.embedder.load_model()

    video_vecs = {}  # {video_id: np.ndarray [1, 512]}
    missing = []

    for vid in tqdm(video_ids_1ka, desc='Encoding 1k-A videos'):
        mp4_path = os.path.join(TEST_VIDEO_DIR, f'{vid}.mp4')
        if not os.path.exists(mp4_path):
            missing.append(vid)
            continue

        frames = extract_uniform_frames(mp4_path, pipeline.embedder.num_frames)
        if frames is None:
            missing.append(vid)
            continue

        # encode_clips expects List[List[frame]] — each inner list = 1 clip's frames
        emb = pipeline.embedder.encode_clips([frames], batch_size=1)  # [1, 512]
        video_vecs[vid] = emb[0]  # [512]

    # 캐시 저장
    with open(EMBED_CACHE, 'wb') as f:
        pickle.dump(video_vecs, f)
    print(f'\n✓ 임베딩 생성 완료: {len(video_vecs)}개 (missing: {len(missing)}개)')
    if missing:
        print(f'  Missing videos: {missing[:10]}{"..." if len(missing) > 10 else ""}')

    # Drive 백업
    DRIVE_ANN = '/content/drive/MyDrive/videorag_prototype/data/msrvtt/annotations'
    os.makedirs(DRIVE_ANN, exist_ok=True)
    shutil.copy2(EMBED_CACHE, os.path.join(DRIVE_ANN, 'test_1ka_embeddings.pkl'))
    print(f'✓ Drive 백업 완료')

else:
    raise FileNotFoundError(
        "1k-A 테스트 영상과 임베딩 캐시 모두 없습니다.\n"
        "아래 중 하나를 준비하세요:\n"
        "  1. TestVideo.zip 다운로드 → data/msrvtt/test_1ka_videos/에 mp4 배치\n"
        "  2. 다른 환경에서 생성한 test_1ka_embeddings.pkl을 annotations/에 복사\n"
        "  자세한 안내: data/msrvtt/README.md"
    )

# ── FlatEvalStore 빌드 (exact brute-force cosine) ──
eval_store = build_flat_index_from_embeddings(video_vecs, dim=512)
print(f'✓ FlatEvalStore 빌드 완료: {eval_store.size}개 벡터 (IndexFlatIP, exact search)')

# 저장 (재실행 시 빠른 로드)
eval_store.save(EVAL_INDEX_PATH)
print(f'✓ Eval 인덱스 저장: {EVAL_INDEX_PATH}')

In [ ]:
# -- Step 5: Tier 1 -- ITM Retrieval on MSR-VTT 1k-A --
#
# [파이프라인]
#   1. 텍스트 1000개 배치 인코딩 -> text_feats_all [1000, 40, 1024]
#   2. ITM 전체 1000개 적용 -> R@1 41.1%
#
# ITC pre-filter(top-128)는 recall@128=77.5%로 정답 손실이 발생해 성능 하락 확인.
# -> full ITM이 최선임을 확정.

import os, time
import torch, torch.nn as nn
import numpy as np
from tqdm import tqdm

iv_model = pipeline.embedder.model
device   = pipeline.embedder.device

# -- ITMScorer 로드 --
ITM_FEAT_PATH = os.path.join(INDEX_DIR, 'itm_vision_features.pt')

if pipeline.itm_scorer is not None:
    print("check pipeline.itm_scorer 사용")
    itm_scorer = pipeline.itm_scorer
elif os.path.exists(ITM_FEAT_PATH):
    from src.phase3_reranking import ITMScorer
    itm_scorer = ITMScorer(iv_model=iv_model, itm_features_path=ITM_FEAT_PATH, device=device)
    print(f"check ITMScorer 직접 로드: {ITM_FEAT_PATH}")
else:
    itm_scorer = None
    print("warn itm_vision_features.pt 없음 -> 영상 직접 인코딩 경로로 진행")

# -- 1. 텍스트 피처 배치 인코딩 [N_txt, 40, 1024] --
captions = [cap for cap, _   in eval_pairs]
gt_vids  = [vid for _,   vid in eval_pairs]

print(f"[1] 텍스트 피처 추출 중... ({len(captions)}개)")
text_feats_all, text_atts_all = [], []
with torch.no_grad():
    for i in tqdm(range(0, len(captions), 64), desc="text encoding"):
        tok = iv_model.tokenizer(
            captions[i:i+64], padding="max_length", truncation=True,
            max_length=iv_model.config.max_txt_l, return_tensors="pt"
        ).to(device)
        feat, _ = iv_model.encode_text(tok)   # [bs, 40, 1024]
        text_feats_all.append(feat.cpu().half())
        text_atts_all.append(tok.attention_mask.cpu())

text_feats_all = torch.cat(text_feats_all)   # [N_txt, 40, 1024]
text_atts_all  = torch.cat(text_atts_all)    # [N_txt, 40]
print(f"   text_feats_all: {tuple(text_feats_all.shape)}")

ordered_vids = list(video_ids_1ka)
vid_to_idx   = {vid: i for i, vid in enumerate(ordered_vids)}

# -- 2. ITM 스코어 행렬 [N_txt, N_vid] --
if itm_scorer is not None:
    print(f"[2] ITMScorer.compute_itm_matrix() -- 전체 1000개")
    t0 = time.time()
    scores = itm_scorer.compute_itm_matrix(
        text_feats_all=text_feats_all,
        text_atts_all=text_atts_all,
        ordered_vids=ordered_vids,
    )
    print(f"   소요: {(time.time()-t0)/60:.1f}분")

else:
    import cv2
    from demo.utils import frames2tensor

    _sd = torch.load(iv_model.config.pretrained_path, map_location="cpu")["module"]
    itm_head = nn.Linear(1024, 2)
    itm_head.weight = nn.Parameter(_sd["itm_head.weight"].float())
    itm_head.bias   = nn.Parameter(_sd["itm_head.bias"].float())
    iv_model.itm_head = itm_head.to(device)
    del _sd
    print("check itm_head 수동 로드 완료")

    enc = iv_model.get_text_encoder()
    model_dtype = next(iv_model.parameters()).dtype

    def _extract_bgr_frames(video_path, num_frames=4):
        cap = cv2.VideoCapture(video_path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total <= 0:
            cap.release(); return None
        frames = []
        for idx in np.linspace(0, total - 1, num_frames, dtype=int):
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret: frames.append(frame)
        cap.release()
        return frames if len(frames) == num_frames else None

    N_txt, N_vid = len(captions), len(ordered_vids)
    print(f"[2] 영상 인코딩 + ITM scoring 중... ({N_vid}개 영상)")

    vis_feats_all = []
    for vid_id in tqdm(ordered_vids, desc="vision encoding"):
        frames = _extract_bgr_frames(os.path.join(TEST_VIDEO_DIR, f"{vid_id}.mp4"), 4)
        if frames is None:
            vis_feats_all.append(torch.zeros(1025, 1408, dtype=torch.float16))
            continue
        t = frames2tensor(frames, fnum=4, target_size=(224, 224), device=device)
        with torch.no_grad():
            vf, _ = iv_model.encode_vision(t, test=True)
        vis_feats_all.append(vf.squeeze(0).cpu().half())
    vis_feats_all = torch.stack(vis_feats_all)

    scores = torch.zeros(N_txt, N_vid)
    BS_ITM = 32
    with torch.no_grad():
        for vi in tqdm(range(N_vid), desc="ITM scoring"):
            vf = vis_feats_all[vi:vi+1].to(device, dtype=model_dtype)
            for ti in range(0, N_txt, BS_ITM):
                tf = text_feats_all[ti:ti+BS_ITM].to(device, dtype=model_dtype)
                ta = text_atts_all[ti:ti+BS_ITM].to(device)
                bs = tf.shape[0]
                out = enc(encoder_embeds=tf, attention_mask=ta,
                          encoder_hidden_states=vf.expand(bs, -1, -1),
                          encoder_attention_mask=None,
                          return_dict=True, mode="fusion")
                s = iv_model.itm_head(out.last_hidden_state[:, 0].float())[:, 1]
                scores[ti:ti+bs, vi] = s.cpu()

# -- 3. R@k 계산 --
ranks = []
for qi, gt_vid in enumerate(gt_vids):
    row      = scores[qi].numpy()
    gt_idx   = vid_to_idx[gt_vid]
    ranks.append(int((row > row[gt_idx]).sum()) + 1)

ranks = np.array(ranks)
r1    = float((ranks <= 1).mean()  * 100)
r5    = float((ranks <= 5).mean()  * 100)
r10   = float((ranks <= 10).mean() * 100)
mdr   = float(np.median(ranks))
mnr   = float(np.mean(ranks))

print(f"{'='*60}")
print(f"  Tier 1 Results: ITM (full 1000)")
print(f"  InternVideo2-1B, #F=4")
print(f"{'='*60}")
print(f"  R@1  = {r1:.1f}%   (paper: 51.9%,  delta: {r1-51.9:+.1f}%)")
print(f"  R@5  = {r5:.1f}%   (paper: 74.6%,  delta: {r5-74.6:+.1f}%)")
print(f"  R@10 = {r10:.1f}%   (paper: 81.7%,  delta: {r10-81.7:+.1f}%)")
print(f"  MdR  = {mdr:.0f}")
print(f"  MnR  = {mnr:.1f}")

tier1_metrics = {
    "method": "Dense+ITM (InternVideo2-1B, #F=4, full 1000)",
    "R@1": r1, "R@5": r5, "R@10": r10, "MdR": mdr, "MnR": mnr,
}
dense_results = [{"r1": float(r<=1), "r5": float(r<=5), "r10": float(r<=10),
                  "rank": int(r), "latency_ms": 0.0} for r in ranks]


In [ ]:
# ── Step 6: Tier 1.5 — Full Pipeline Latency Ablation (Production Corpus) ──
#
# Production 인덱스(1000 clips, 1k-A test set)에서 4가지 검색 설정의 레이턴시 비교.
# 데모 쿼리 사용 — Ground Truth 없이 순수 속도 프로파일링.
#
# 설정:
#   A. BM25 only
#   B. Dense only (InternVideo2, IVFFlat production index)
#   C. Hybrid (BM25 + Dense + WRRF w_visual=0.6, w_text=0.4, k=60)
#   D. Hybrid + ColBERT reranking (full pipeline)

import pandas as pd

if not HAS_PRODUCTION_INDEX:
    print('⚠ Production 인덱스 없음 — Tier 1.5 스킵')
    tier15_df = None
else:
    # 데모 쿼리 로드
    DEMO_QUERIES_PATH = os.path.join(PROJECT_ROOT, 'data', 'queries', 'demo_queries.json')
    if os.path.exists(DEMO_QUERIES_PATH):
        with open(DEMO_QUERIES_PATH) as f:
            demo_queries = [q['query'] for q in json.load(f)]
    else:
        demo_queries = [
            'a man playing guitar on stage',
            'people cooking food in a kitchen',
            'dog running in a park',
            'person dancing to music',
            'cars driving on a highway',
            'children playing soccer',
            'man giving a presentation',
            'cat sitting on a couch',
        ]
    print(f'Tier 1.5: Latency Ablation — {len(demo_queries)} queries on production corpus')
    print()

    configs = {
        'BM25 only': 'bm25',
        'Dense only': 'dense',
        'Hybrid (WRRF)': 'hybrid',
        'Hybrid + ColBERT': 'full',
    }

    latency_records = []

    for config_name, mode in configs.items():
        query_latencies = []

        for query in demo_queries:
            t0 = time.perf_counter()

            if mode == 'bm25':
                pipeline.bm25.search(query, top_n=10)

            elif mode == 'dense':
                pipeline.dense_retriever.encode_and_search(query, top_k=10)

            elif mode == 'hybrid':
                bm25_res = pipeline.bm25.search(query, top_n=100)
                _, dense_res = pipeline.dense_retriever.encode_and_search(query, top_k=100)
                pipeline.hybrid_fusion.fuse(bm25_res, dense_res)

            elif mode == 'full':
                pipeline.search(query, top_k=10, assemble_video=False)

            elapsed = (time.perf_counter() - t0) * 1000
            query_latencies.append(elapsed)

        avg = np.mean(query_latencies)
        p95 = np.percentile(query_latencies, 95)
        latency_records.append({
            'Configuration': config_name,
            'Avg Latency (ms)': round(avg, 0),
            'P95 Latency (ms)': round(p95, 0),
            'Min (ms)': round(min(query_latencies), 0),
            'Max (ms)': round(max(query_latencies), 0),
        })
        print(f'  {config_name:20s}  avg={avg:.0f}ms  p95={p95:.0f}ms')

    tier15_df = pd.DataFrame(latency_records)
    print()
    print(tier15_df.to_string(index=False))
    print('\n✓ Tier 1.5 완료')

In [ ]:
# ── Step 7: 결과 종합 + 비교 테이블 ──
import pandas as pd

# ── Tier 1 결과 테이블 ──
paper_baseline = {
    'Method': 'InternVideo2-1B #F=4 (paper)',
    'R@1': 51.9, 'R@5': 74.6, 'R@10': 81.7,
    'MdR': '-', 'MnR': '-',
}

our_dense = {
    'Method': tier1_metrics['method'],
    'R@1': tier1_metrics['R@1'],
    'R@5': tier1_metrics['R@5'],
    'R@10': tier1_metrics['R@10'],
    'MdR': tier1_metrics['MdR'],
    'MnR': tier1_metrics['MnR'],
}

tier1_df = pd.DataFrame([paper_baseline, our_dense])

print('=' * 70)
print('  Table 1: MSR-VTT 1k-A Zero-Shot Text-to-Video Retrieval')
print('=' * 70)
print(tier1_df.to_string(index=False))
print()

# ── Tier 1.5 latency 테이블 (있으면) ──
if tier15_df is not None:
    print('=' * 70)
    print('  Table 2: Pipeline Latency Ablation (Production Corpus)')
    print('=' * 70)
    print(tier15_df.to_string(index=False))
    print()

# ── CSV 저장 ──
tier1_csv = os.path.join(OUTPUT_DIR, 'tier1_retrieval_results.csv')
tier1_df.to_csv(tier1_csv, index=False)
print(f'✓ Tier 1 CSV: {tier1_csv}')

if tier15_df is not None:
    tier15_csv = os.path.join(OUTPUT_DIR, 'tier15_latency_results.csv')
    tier15_df.to_csv(tier15_csv, index=False)
    print(f'✓ Tier 1.5 CSV: {tier15_csv}')

# ── Markdown 저장 (README 삽입용) ──
tier1_md = os.path.join(OUTPUT_DIR, 'tier1_retrieval_results.md')
with open(tier1_md, 'w') as f:
    f.write('## MSR-VTT 1k-A Zero-Shot Text-to-Video Retrieval\n\n')
    f.write('| Method | R@1 | R@5 | R@10 | MdR | MnR |\n')
    f.write('|---|---|---|---|---|---|\n')
    for _, row in tier1_df.iterrows():
        f.write(f"| {row['Method']} | {row['R@1']} | {row['R@5']} | {row['R@10']} | {row['MdR']} | {row['MnR']} |\n")
    f.write('\n')
    if tier15_df is not None:
        f.write('## Pipeline Latency Ablation\n\n')
        f.write(tier15_df.to_markdown(index=False))
        f.write('\n')
print(f'✓ Markdown: {tier1_md}')
print('\n✓ 결과 종합 완료')

In [ ]:
# ── Step 8: 시각화 ──
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

fig, axes = plt.subplots(1, 3 if tier15_df is not None else 2, figsize=(18, 5))

# ── (1) Tier 1: R@K 비교 (ours vs paper) ──
ax = axes[0]
x = np.arange(3)
width = 0.35
paper_vals = [51.9, 74.6, 81.7]
our_vals   = [tier1_metrics['R@1'], tier1_metrics['R@5'], tier1_metrics['R@10']]

bars1 = ax.bar(x - width/2, paper_vals, width, label='Paper (Table 24a)', color='#4e79a7', alpha=0.8)
bars2 = ax.bar(x + width/2, our_vals,   width, label='Ours (Dense-only)', color='#59a14f', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(['R@1', 'R@5', 'R@10'])
ax.set_ylabel('Recall (%)')
ax.set_title('Tier 1: MSR-VTT 1k-A Zero-Shot T2V')
ax.legend(loc='lower right')
ax.set_ylim(0, 100)

# 값 표시
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)

# ── (2) Tier 1: Rank 분포 히스토그램 ──
ax = axes[1]
ax.hist(ranks, bins=50, color='#f28e2b', alpha=0.8, edgecolor='white')
ax.axvline(x=mdr, color='red', linestyle='--', label=f'Median={mdr:.0f}')
ax.axvline(x=mnr, color='blue', linestyle='--', label=f'Mean={mnr:.1f}')
ax.set_xlabel('Rank (1-indexed)')
ax.set_ylabel('Count')
ax.set_title('Tier 1: Rank Distribution (1000 queries)')
ax.legend()

# ── (3) Tier 1.5: Latency 비교 (있으면) ──
if tier15_df is not None:
    ax = axes[2]
    config_names = tier15_df['Configuration'].values
    avg_lats = tier15_df['Avg Latency (ms)'].values
    colors = ['#4e79a7', '#59a14f', '#f28e2b', '#e15759']
    bars = ax.barh(config_names, avg_lats, color=colors, alpha=0.8)
    ax.set_xlabel('Avg Latency (ms)')
    ax.set_title('Tier 1.5: Pipeline Latency Ablation')
    for bar, val in zip(bars, avg_lats):
        ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
                f'{val:.0f}ms', va='center', fontsize=10)

plt.tight_layout()
chart_path = os.path.join(OUTPUT_DIR, 'evaluation_report.png')
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
print(f'✓ 차트 저장: {chart_path}')
plt.show()

In [ ]:
# ── Step 9: JSON 리포트 저장 + Drive 백업 ──

report = {
    'eval_protocol': {
        'benchmark': 'MSR-VTT 1k-A (JSFusion split)',
        'corpus_size': eval_store.size,
        'n_queries': len(eval_pairs),
        'model': pipeline.embedder.model_name,
        'num_frames': pipeline.embedder.num_frames,
        'embed_dim': 512,
        'index_type': 'IndexFlatIP (exact brute-force)',
    },
    'tier1_dense_only': tier1_metrics,
    'paper_baseline': {
        'source': 'InternVideo2 Table 24a (Supplementary)',
        'model': 'InternVideo2s2-1B',
        'frames': 4,
        'R@1': 51.9, 'R@5': 74.6, 'R@10': 81.7,
    },
    'parity_check': parity_report,
}

if tier15_df is not None:
    report['tier15_latency'] = tier15_df.to_dict(orient='records')

# per-query 결과도 포함 (디버깅용)
report['tier1_per_query'] = dense_results

report_path = os.path.join(OUTPUT_DIR, 'evaluation_report.json')
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=str)
print(f'✓ 리포트 저장: {report_path}')

# Drive 백업
DRIVE_OUTPUT = '/content/drive/MyDrive/videorag_prototype/output'
if os.path.exists('/content/drive'):
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)
    for fname in ['evaluation_report.json', 'evaluation_report.png',
                  'tier1_retrieval_results.csv', 'tier1_retrieval_results.md']:
        src = os.path.join(OUTPUT_DIR, fname)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(DRIVE_OUTPUT, fname))
    if tier15_df is not None:
        src = os.path.join(OUTPUT_DIR, 'tier15_latency_results.csv')
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(DRIVE_OUTPUT, 'tier15_latency_results.csv'))
    print(f'✓ Drive 백업 완료: {DRIVE_OUTPUT}')

print('\n✅ Evaluation 완료!')

## Interpretation Guide

### Tier 1: Dense-only vs Paper

| Delta | 해석 |
|---|---|
| ±0-2% | 논문 재현 성공. Video decoder, frame sampling 미세 차이 |
| ±2-5% | 허용 범위. README footnote로 원인 명시 |
| >5% | 셋팅 점검 필요 — Parity Check 항목 재확인 |

**우리 Dense-only가 논문과 유사하면**: InternVideo2를 올바르게 통합했다는 증거.
이 위에 BM25, WRRF, ColBERT을 쌓아 올린 게 우리 파이프라인의 엔지니어링 기여.

### Tier 1.5: Latency Ablation

- **BM25 only**: 텍스트 검색 (spaCy lemmatizer). 가장 빠르지만 시맨틱 이해 제한
- **Dense only**: InternVideo2 인코딩 + FAISS IVFFlat. 시맨틱 강점, 인코딩 비용 존재
- **Hybrid (WRRF)**: BM25 + Dense 결합. w_visual=0.6, w_text=0.4, k=60
- **Hybrid + ColBERT**: 최종 파이프라인. reranking으로 정밀도 향상, 레이턴시 증가

### Footnotes

- Eval 인덱스: `IndexFlatIP` (exact brute-force), production은 `IVFFlat` (approximate)
- 논문 미보고 항목: MdR, MnR은 논문 Table 24a에 없음 (우리만의 추가 지표)
- per-query 결과는 `output/evaluation_report.json`에 포함